# 주택담보대출 월 상환액 계산기

대출 조건에 따른 원리금균등·원금균등·만기일시 상환 결과를 비교합니다.

In [ ]:
# @title 대출 조건을 입력하고 실행하세요
대출금액_억원 = 1.0  # @param {type:"number", min:0.01, step:0.1}
대출기간_년 = 30  # @param {type:"integer", min:1, step:1}
연이율_퍼센트 = 4.5  # @param {type:"number", min:0, step:0.1}
연도별_상환방식 = '원리금균등상환'  # @param ['원리금균등상환', '원금균등상환', '만기일시상환']

import pandas as pd
from IPython.display import HTML, Markdown, display

principal = float(대출금액_억원) * 100_000_000
years = int(대출기간_년)
annual_rate = float(연이율_퍼센트)

if principal <= 0:
    raise ValueError('대출금액은 0보다 커야 합니다.')
if years <= 0:
    raise ValueError('대출기간은 1년 이상이어야 합니다.')
if annual_rate < 0:
    raise ValueError('연이율은 0% 이상이어야 합니다.')

months = years * 12
monthly_rate = annual_rate / 100 / 12

def format_korean_currency(amount):
    amount = int(round(amount))
    if amount == 0:
        return '0원'
    eok, remainder = divmod(amount, 100_000_000)
    man, won = divmod(remainder, 10_000)
    parts = []
    if eok:
        parts.append(f'{eok:,}억')
    if man:
        parts.append(f'{man:,}만')
    if won:
        parts.append(f'{won:,}')
    return ' '.join(parts) + '원'

def equal_principal_and_interest():
    if monthly_rate == 0:
        monthly_payment = principal / months
    else:
        compound = (1 + monthly_rate) ** months
        monthly_payment = principal * monthly_rate * compound / (compound - 1)

    balance = principal
    total_interest = 0.0
    last_payment = monthly_payment
    annual_schedule = []
    yearly_payment = yearly_principal = yearly_interest = 0.0
    for month in range(1, months + 1):
        interest = balance * monthly_rate
        principal_payment = monthly_payment - interest
        actual_payment = monthly_payment
        if month == months:
            principal_payment = balance
            last_payment = principal_payment + interest
            actual_payment = last_payment
        balance -= principal_payment
        total_interest += interest
        yearly_payment += actual_payment
        yearly_principal += principal_payment
        yearly_interest += interest

        if month % 12 == 0:
            annual_schedule.append({
                '연차': month // 12,
                '연간 납입액': round(yearly_payment),
                '상환 원금': round(yearly_principal),
                '납부 이자': round(yearly_interest),
                '대출 잔액': round(max(balance, 0)),
            })
            yearly_payment = yearly_principal = yearly_interest = 0.0

    summary = {
        '상환 방식': '원리금균등상환',
        '첫 달 납입액': round(monthly_payment),
        '마지막 달 납입액': round(last_payment),
        '총 납부 이자': round(total_interest),
        '총 상환 금액': round(principal + total_interest),
    }
    return summary, annual_schedule

def equal_principal():
    monthly_principal = principal / months
    balance = principal
    total_interest = 0.0
    annual_schedule = []
    yearly_payment = yearly_principal = yearly_interest = 0.0
    first_payment = last_payment = 0.0
    for month in range(1, months + 1):
        interest = balance * monthly_rate
        principal_payment = balance if month == months else monthly_principal
        payment = principal_payment + interest
        if month == 1:
            first_payment = payment
        if month == months:
            last_payment = payment
        balance -= principal_payment
        total_interest += interest
        yearly_payment += payment
        yearly_principal += principal_payment
        yearly_interest += interest

        if month % 12 == 0:
            annual_schedule.append({
                '연차': month // 12,
                '연간 납입액': round(yearly_payment),
                '상환 원금': round(yearly_principal),
                '납부 이자': round(yearly_interest),
                '대출 잔액': round(max(balance, 0)),
            })
            yearly_payment = yearly_principal = yearly_interest = 0.0

    summary = {
        '상환 방식': '원금균등상환',
        '첫 달 납입액': round(first_payment),
        '마지막 달 납입액': round(last_payment),
        '총 납부 이자': round(total_interest),
        '총 상환 금액': round(principal + total_interest),
    }
    return summary, annual_schedule

def bullet_maturity():
    monthly_interest = principal * monthly_rate
    total_interest = monthly_interest * months
    annual_schedule = []
    for year in range(1, years + 1):
        is_last_year = year == years
        annual_principal = principal if is_last_year else 0
        annual_interest = monthly_interest * 12
        annual_schedule.append({
            '연차': year,
            '연간 납입액': round(annual_principal + annual_interest),
            '상환 원금': round(annual_principal),
            '납부 이자': round(annual_interest),
            '대출 잔액': 0 if is_last_year else round(principal),
        })

    summary = {
        '상환 방식': '만기일시상환',
        '첫 달 납입액': round(monthly_interest),
        '마지막 달 납입액': round(principal + monthly_interest),
        '총 납부 이자': round(total_interest),
        '총 상환 금액': round(principal + total_interest),
    }
    return summary, annual_schedule

equal_summary, equal_annual = equal_principal_and_interest()
principal_summary, principal_annual = equal_principal()
bullet_summary, bullet_annual = bullet_maturity()
results = [
    equal_summary,
    principal_summary,
    bullet_summary,
]
annual_results = {
    '원리금균등상환': (equal_summary, equal_annual),
    '원금균등상환': (principal_summary, principal_annual),
    '만기일시상환': (bullet_summary, bullet_annual),
}
selected_summary, annual_schedule = annual_results[연도별_상환방식]
result_table = pd.DataFrame(results)
result_display = result_table.copy()
for column in result_display.columns[1:]:
    result_display[column] = result_display[column].map(
        format_korean_currency
    )

TABLE_STYLE = '''
<style>
.loan-table-wrap {
    max-width: 700px;
    margin: 12px 0 28px;
    overflow-x: auto;
    border: 1px solid #f0f2f5;
    border-radius: 12px;
}
.loan-table {
    width: 100%;
    border-collapse: separate;
    border-spacing: 0;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
    font-size: 13px;
    font-variant-numeric: tabular-nums;
    color: #1e293b;
}
.loan-table thead th {
    position: sticky;
    top: 0;
    z-index: 1;
    padding: 8px 12px;
    background: #2b4a75;
    border-bottom: 2px solid #7fb3d5;
    color: white;
    font-weight: 700;
    text-align: right;
    white-space: nowrap;
}
.loan-table tbody td {
    padding: 7px 12px;
    border-bottom: 1px solid #f0f2f5;
    background: white;
    text-align: right;
    white-space: nowrap;
}
.loan-table tbody tr:nth-child(even) td { background: #fafbfc; }
.loan-table tbody tr:hover td { background: #eff6ff; }
.loan-table tbody tr:last-child td { border-bottom: 0; }
.loan-table thead th:first-child { text-align: center; }
.loan-table tbody td:first-child { text-align: center; }
.summary-table tbody td:first-child {
    color: #1e293b;
    font-weight: 600;
}
.annual-table tbody td:first-child {
    color: #6b7280;
    font-weight: 400;
}
.loan-table tbody td:last-child { color: #0f172a; font-weight: 400; }
.annual-table { table-layout: fixed; }
.annual-table th:nth-child(1), .annual-table td:nth-child(1) { width: 9%; }
.annual-table th:nth-child(2), .annual-table td:nth-child(2) { width: 20%; }
.annual-table th:nth-child(3), .annual-table td:nth-child(3) { width: 22%; }
.annual-table th:nth-child(4), .annual-table td:nth-child(4) { width: 22%; }
.annual-table th:nth-child(5), .annual-table td:nth-child(5) { width: 27%; }
.annual-table tbody tr:last-child td {
    background: #e8edf3;
    border-top: 2px solid #94a3b8;
    color: #0f172a;
    font-weight: 700;
}
.table-title {
    margin: 30px 0 8px;
    color: #0f172a;
    font-size: 20px;
    font-weight: 700;
}
.table-caption {
    margin: 0 0 10px;
    color: #64748b;
    font-size: 12px;
}
</style>
'''
display(HTML(TABLE_STYLE))

display(Markdown(
    f'## 계산 결과\n'
    f'- 대출금액: **{format_korean_currency(principal)}**\n'
    f'- 대출기간: **{years}년 ({months:,}개월)**\n'
    f'- 연이율: **{annual_rate:g}%**'
))
summary_html = result_display.to_html(
    index=False, border=0, classes='loan-table summary-table'
)
display(HTML(f'<div class="loan-table-wrap">{summary_html}</div>'))

annual_table = pd.DataFrame(annual_schedule)
annual_total = pd.DataFrame([{
    '연차': f'합계 ({years}년)',
    '연간 납입액': selected_summary['총 상환 금액'],
    '상환 원금': round(principal),
    '납부 이자': selected_summary['총 납부 이자'],
    '대출 잔액': annual_schedule[-1]['대출 잔액'],
}])
annual_display = pd.concat([annual_table, annual_total], ignore_index=True)
for column in annual_display.columns[1:]:
    annual_display[column] = annual_display[column].map(
        format_korean_currency
    )
display(HTML(
    f'<div class="table-title">{연도별_상환방식} 연도별 내역</div>'
    f'<div class="table-caption">{연도별_상환방식} │ 대출 '
    f'{format_korean_currency(principal)} │ 연 {annual_rate:g}% │ {years}년</div>'
))
annual_html = annual_display.to_html(
    index=False, border=0, classes='loan-table annual-table'
)
display(HTML(f'<div class="loan-table-wrap">{annual_html}</div>'))

monthly_payment = results[0]['첫 달 납입액']
per_eok_payment = monthly_payment / 대출금액_억원
display(Markdown(
    f'### 한 줄 결론\n'
    f'**{대출금액_억원:g}억 원**을 **{years}년**, **연 {annual_rate:g}%**, '
    f'**원리금균등상환**으로 빌리면 월 상환액은 약 '
    f'**{format_korean_currency(monthly_payment)}**입니다.  '
    f'1억 원당 월 상환액으로 환산하면 약 '
    f'**{format_korean_currency(per_eok_payment)}**입니다.'
))